<a href="https://colab.research.google.com/github/jaewoenlee/ML/blob/main/ch21_colab_English_ex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Open this notebook in [Google Colab](https://colab.research.google.com/) using **File → Upload notebook** and select `ch21_colab_English.ipynb`.


# Chapter 21. Natural Language Processing with Deep Learning

## 1. Text Tokenization

In [ ]:
# 변경 사항: 토큰화, 원-핫 인코딩, 감성 분류 예문을 영어로 바꾸고 출력 문구를 영어로 통일했습니다.
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Flatten,Embedding
from tensorflow.keras.utils import to_categorical
from numpy import array

# Import text_to_word_sequence from Keras's text preprocessing functions.
from tensorflow.keras.preprocessing.text import text_to_word_sequence

# Specify the text to preprocess.
text = 'You never know until you try.'

# Tokenize the text.
result = text_to_word_sequence(text)
print("\nOriginal text:\n", text)
print("\nTokenized text:\n", result)

In [ ]:
# Count word frequencies.

# Specify three sentences to preprocess.
docs = ['First, we split text into words.',
       'We tokenize text into words for deep learning.',
       'Tokenized text can be used for deep learning.',
       ]

# Preprocess the sentences using the tokenizer.
token = Tokenizer()            # Initialize the tokenizer.
token.fit_on_texts(docs)       # Apply the tokenizer to the sentences.

# Print the calculated word frequencies using the different options.
# Tokenizer()'s word_counts uses the OrderedDict class, which preserves order.
print("\nWord counts:\n", token.word_counts)

# The output order is random.
print("\nDocument count: ", token.document_count)
print("\nNumber of documents containing each word:\n", token.word_docs)
print("\nWord indices:\n",  token.word_index)

## 2. One-Hot Encoding of Words

In [ ]:
text="We learn by doing and we grow by learning"
token = Tokenizer()
token.fit_on_texts([text])
print(token.word_index)

In [ ]:
x=token.texts_to_sequences([text])
print(x)

In [ ]:
# Add one to the number of indices to create the one-hot encoding array.
word_size = len(token.word_index) + 1
x = to_categorical(x, num_classes=word_size)
print(x)

## 4. Reading Text to Predict Positive or Negative Sentiment

In [ ]:
# Specify the text reviews.
docs = [
    "Really fun",
    "Excellent",
    "I enjoyed the movie",
    "Highly recommended",
    "Worth watching again",
    "Not impressed",
    "Very disappointing",
    "The movie was boring",
    "The acting was awkward",
    "Not fun"
]

# Label positive reviews as 1 and negative reviews as 0.
classes = array([1,1,1,1,1,0,0,0,0,0])

# Tokenization
token = Tokenizer()
token.fit_on_texts(docs)
print(token.word_index)


In [ ]:
x = token.texts_to_sequences(docs)
print("\nTokenized reviews:\n",  x)

In [ ]:
# Pad sequences of different lengths to a length of 4.
padded_x = pad_sequences(x, 4)
print("\nPadded sequences:\n", padded_x)

In [ ]:
# Specify the number of words to pass to the embedding layer.
word_size = len(token.word_index) +1

# Build a deep learning model with word embeddings and print the results.
model = Sequential()
model.add(Embedding(word_size, 8))
model.build((None, 4))
model.add(Flatten())
model.add(Dense(1, activation='sigmoid'))
model.summary()

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(padded_x, classes, epochs=20)
print("\n Accuracy: %.4f" % (model.evaluate(padded_x, classes)[1]))

In [ ]:
# Specify a new review (all words appeared in the training vocabulary).
new_docs = ["Really enjoyed the movie"]

# Convert the review using the tokenizer fitted on the training reviews.
new_x = token.texts_to_sequences(new_docs)

# Pad to the same length used during training.
new_padded = pad_sequences(new_x, maxlen=4)

# Get the predicted positive probability for the first review.
p = float(model.predict(new_padded, verbose=0)[0, 0])

print("Input review:", new_docs[0])
print(f"Positive probability: {p:.3f}")
print("Prediction:", "Positive" if p > 0.5 else "Negative")
